In [ ]:
# --- repo bootstrap: make src/ importable and run from repo root (works wherever the kernel starts) ---
import sys, os
from pathlib import Path
_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'src'))
os.chdir(_ROOT)

# Aave V3.1 — data validation checks

Runs **per-table** validation on the CSVs in `query_result_data/` and saves, to
`validation_results/`: **per-table result CSVs** `<table>__<check>.csv` (summary, null_counts,
type_checks, ge_results) **plus one combined Markdown report** `validation_report.md`
(an overview PASS/FAIL table at the top, then one section per table).

**Kernel:** pick **Python (aave-ge)** — the isolated venv with Great Expectations 0.18
(pandas 2.3.3). Your main venv stays on pandas 3.0; only this notebook uses the GE venv.
*If the GE import fails or nothing saves, you are on the wrong kernel.*

**Flow per table:** pandas null / type / duplicate checks first, then the Great Expectations
suite. Edit `BATCH` to run a few *similar-schema* tables at a time (don't mix all schemas).
Each run overwrites the report and the per-table CSVs.

Libraries used: `pandas`, `IPython.display.display`, stdlib `glob`/`pathlib`, local `data_validation`.

In [2]:
import glob
from pathlib import Path
import pandas as pd
from IPython.display import display
import data_validation as dv
import adv_validation as adv


DATA_DIR = "query_result_data"
RESULTS_DIR = "validation_results"

In [3]:
# See the available tables (label <- file) so you can choose a batch
# for f in sorted(glob.glob(f"{DATA_DIR}/*.csv")):
#     print(f"{dv.table_name_from_path(f):20s} <- {Path(f).name}")

for f in sorted(glob.glob(f"{DATA_DIR}/*.csv")):
    print(f" {Path(f).name}")

 borrow_repay_7798273_20260624T081133Z.csv
 collateral_toggle_7798372_20260624T082649Z.csv
 decimal_reference_part1_7711171_20260623T185103Z.csv
 decimal_reference_part1_7711171_20260623T190322Z.csv
 decimal_reference_part2_7711265_20260623T191755Z.csv
 decimal_reference_part3_7711276_20260623T192309Z.csv
 decimal_reference_part4_7711290_20260623T192311Z.csv
 decimal_reference_part5_7711298_20260623T192320Z.csv
 decimal_reference_part6_7711304_20260623T192322Z.csv
 decimal_reference_part9_collateral_toggle_7711307_20260623T192459Z.csv
 flashloan_7798349_20260624T082343Z.csv
 liquidation_7798339_20260624T082345Z.csv
 oracle_price_usd_eth_weth_6h_7798226_20260624T092854Z.csv
 reserve_config_7804264_20260624T204732Z.csv
 reserve_state_rates_7711042_20260624T074536Z.csv
 supply_withdraw_7702138_20260624T074220Z.csv
 user_account_7798351_20260624T082341Z.csv


In [ ]:
# Pick a few SIMILAR-schema tables to validate (edit this list; don't run all schemas at once)
BATCH = [
    # "borrow_repay_7798273_20260624T081133Z.csv",
    # "collateral_toggle_7798372_20260624T082649Z.csv",
    # "decimal_reference_part1_7711171_20260623T185103Z.csv",
    # "decimal_reference_part1_7711171_20260623T190322Z.csv",
    # "decimal_reference_part2_7711265_20260623T191755Z.csv",
    # "decimal_reference_part3_7711276_20260623T192309Z.csv",
    # "decimal_reference_part4_7711290_20260623T192311Z.csv",
    # "decimal_reference_part5_7711298_20260623T192320Z.csv",
    # "decimal_reference_part6_7711304_20260623T192322Z.csv",
    # "decimal_reference_part9_collateral_toggle_7711307_20260623T192459Z.csv",
    # "flashloan_7798349_20260624T082343Z.csv",
    # "liquidation_7798339_20260624T082345Z.csv",
    # "oracle_price_usd_eth_weth_6h_7798226_20260624T092854Z.csv",
    # "reserve_config_7804264_20260624T204732Z.csv",
    "reserve_state_rates_7711042_20260624T074536Z.csv",
    # "supply_withdraw_7702138_20260624T074220Z.csv",
    # "user_account_7798351_20260624T082341Z.csv",
]

# pandas checks first, then great expectations.
# saves per-table result CSVs (<table>__<check>.csv) AND one combined report
batch = dv.validate_batch([f"{DATA_DIR}/{f}" for f in BATCH], RESULTS_DIR)
print(f"saved combined report -> {batch['report_path']}")
# print(f"saved {len(batch['csv_paths'])} per-table result CSVs:")
# for p in batch["csv_paths"]:
#     print(f"  {p}")
# display(batch["overview"])

# per-table tables for inline review
for res in batch["results"]:
    print(f"\n===== {res['table']} =====")
    display(res["null_counts"])
    # display(res["ge_results"])

---

## Transformed-frame validation — model-ready frames (`transformed_data/`)

Validates the two post-transform frames against **`context/data_val.md`** (Tiers 0–4),
returning **numeric** results — pass-rate %, counts, and the offending columns / `time_bucket`s.

| Frame | Grain | Key |
|-------|-------|-----|
| `DF_common_final` (368×46) | protocol-level 6h feature matrix | `time_bucket` (PK) |
| `DF_common_1` (6164×9) | asset-level reserve panel | `(time_bucket, asset)` |

Checks are **consolidated**: one row per *logical* check scanning all relevant columns, with
offenders listed in `anomaly_columns` — so a sub-100 % pass rate stands out instead of drowning
in per-column green. The suite is meant to **surface real imperfections, not rubber-stamp the
data** — several checks are expected to fail here:

- **`bps fields on bps scale`** (T1) — `avg_ltv` / `avg_current_liquidation_threshold` were silently rescaled to a fraction (0–1) instead of bps (0–10000).
- **`activity counts complete`** (T0) — quiet 6h buckets were left *null* instead of `0` (a left-join gap, not a real zero).
- **`liquidation zero-coupling` / `economics`** (T2) and **single-user `avg_*` sampling** (T2, soft).

Logic lives in `adv_validation.py` (`tf_tier0_schema` … `tf_tier4_temporal`, plus
`validate_transformed_final` / `validate_reserve_panel`); each cell below shows one tier.
**Kernel:** `Python (aave-ge)` — pandas only, no GE needed.

Each result row: `tier, check, columns, severity, n_checked, n_pass, n_fail, pass_rate_pct,
anomaly_columns, anomaly_buckets, detail`. `severity` is **hard** (invariant / data bug) or
**soft** (advisory — flag & investigate).

In [ ]:
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 70)

TRANSFORMED_DIR = "transformed_data"
df_final = pd.read_csv(f"{TRANSFORMED_DIR}/DF_common_final.csv")   # protocol-level, time_bucket PK
df_panel = pd.read_csv(f"{TRANSFORMED_DIR}/DF_common_1.csv")       # asset-level reserve panel
print(f"DF_common_final {df_final.shape}  |  DF_common_1 {df_panel.shape}")

# null + zero counts per numeric column of DF_common_final (via adv_validation.statistical_validation)
nz = adv.statistical_validation(df_final, table_name="DF_common_final", save=False)
display(nz[["column", "n_null", "null_pct", "n_zero", "zero_pct"]])

DF_common_final (368, 46)  |  DF_common_1 (6164, 9)


,column,n_null,null_pct,n_zero,zero_pct
0,supply_tx_count,0,0.0000,0,0.0000
1,withdrawal_tx_count,0,0.0000,0,0.0000
2,unique_suppliers,0,0.0000,0,0.0000
3,unique_withdraw_users,0,0.0000,0,0.0000
4,supply_amount_value_usd,0,0.0000,0,0.0000
5,supply_amount_value_eth,0,0.0000,0,0.0000
6,withdrawal_amount_value_usd,0,0.0000,0,0.0000
7,withdrawal_amount_value_eth,0,0.0000,0,0.0000
8,borrow_tx_count,0,0.0000,0,0.0000
9,repay_tx_count,0,0.0000,0,0.0000


### Tier 0 — Schema & type
`time_bucket` is a tz-explicit UTC timestamp; counts are integer-valued; `*_value_*` / `avg_*`
parse as numeric floats. **Hard fail here:** *activity counts complete* — `liquidation_*` /
`as_*` are null in 173/368 buckets and the user-account counts in 79, because quiet 6h buckets
were never `0`-filled (a left-join gap, `data_val.md` Tier 4.5). Offending columns + per-column
null counts are in `anomaly_columns` / `detail`.

In [19]:
display(adv.tf_tier0_schema(df_final))

,tier,check,columns,severity,n_checked,n_pass,n_fail,pass_rate_pct,anomaly_columns,anomaly_buckets,detail
0,T0,time_bucket is a tz-explicit UTC timestamp,time_bucket,hard,368,368,0,100.0000,,,
1,T0,counts are integer-valued (no fractional aggregation),24 count columns,hard,7809,7809,0,100.0000,,,
2,T0,"activity counts complete (quiet bucket = 0, not null)",24 count columns,hard,8832,7809,1023,88.4171,as_collateral_tx_count;as_debt_tx_count;liquidation_tx_count;uniqu...,,null cells per column (should be 0-filled): as_collateral_tx_count...
3,T0,value / avg columns parse as a numeric float,21 value/avg columns,hard,6641,6641,0,100.0000,,,


flagge columns are as_collateral_tx_count;as_debt_tx_count;liquidation_tx_count;uniqu

In [ ]:
#

### Tier 1 — Univariate domain / range
Non-negativity (counts / `*_value_*` / `avg_*_base`); bps numeric bound `[0, 10000]`; sentinel /
un-scaled-wei scan. **Hard fail here:** *bps fields on bps scale* — `avg_ltv` &
`avg_current_liquidation_threshold` are stored as **fractions (0–0.95)**, so ~99 % of values sit
in `(0, 1]`. They pass the `[0, 10000]` bound only because `0.79 < 10000`; this check catches the
silent bps→fraction rescale (`data_val.md` Tier 1.2).

In [20]:
display(adv.tf_tier1_domain(df_final))

,tier,check,columns,severity,n_checked,n_pass,n_fail,pass_rate_pct,anomaly_columns,anomaly_buckets,detail
0,T1,"non-negative (>= 0): counts, *_value_*, avg_*_base",43 columns,hard,13872,13872,0,100.000,,,
1,T1,"bps numeric bound [0, 10000]","avg_ltv, avg_current_liquidation_threshold",hard,578,578,0,100.000,,,
2,T1,bps fields on bps scale (not silently rescaled to fraction),"avg_ltv, avg_current_liquidation_threshold",hard,578,4,574,0.692,avg_ltv;avg_current_liquidation_threshold,,"fractional (0,1] values per column — spec expects bps 0-10000: avg..."
3,T1,no sentinel / un-scaled wei magnitude in value fields,19 value/base columns,hard,6063,6063,0,100.000,,,"flags (-1, 1e+18, 999999999) or |v|>=1e+15; offenders: none"


### Tier 2 — Intra-row logical invariants
`unique ≤ tx_count` and subset/mode counts (consolidated, one row each); **zero-coupling** per
family (`count ⇔ amount ⇔ uniques`, `detail` separates *structural zero* from *all-null quiet/gap*).
**Fails here:** liquidation zero-coupling (1 bucket, hard) and liquidation economics (collateral <
debt covered, 2 buckets, soft). Plus risk ordering `avg_ltv ≤ avg_current_liquidation_threshold`,
solvency (soft), sampling consistency, and a **soft** flag that `avg_*` is backed by **≥ 2 sampled
users** — ~34 % of buckets have `sampled_user_count == 1`, making those `avg_*` a single user's value.

In [21]:
display(adv.tf_tier2_invariants(df_final))

,tier,check,columns,severity,n_checked,n_pass,n_fail,pass_rate_pct,anomaly_columns,anomaly_buckets,detail
0,T2,unique users <= tx count (all families),9 unique/tx family pairs,hard,2966,2966,0,100.0000,,,
1,T2,subset count <= total (mode partitions),3 subset pairs,hard,1104,1104,0,100.0000,,,
2,T2,supply: zero-coupling (count<=>amount<=>uniques),supply_tx_count,hard,368,368,0,100.0000,,,all_null(quiet/gap)=0 structural_zero=0 active=368
3,T2,withdrawal: zero-coupling (count<=>amount<=>uniques),withdrawal_tx_count,hard,368,368,0,100.0000,,,all_null(quiet/gap)=0 structural_zero=0 active=368
4,T2,borrow: zero-coupling (count<=>amount<=>uniques),borrow_tx_count,hard,368,368,0,100.0000,,,all_null(quiet/gap)=0 structural_zero=0 active=368
5,T2,repay: zero-coupling (count<=>amount<=>uniques),repay_tx_count,hard,368,368,0,100.0000,,,all_null(quiet/gap)=0 structural_zero=0 active=368
6,T2,liquidation: zero-coupling (count<=>amount<=>uniques),liquidation_tx_count,hard,195,194,1,99.4872,"liquidation_tx_count,liquidated_collateral_value_usd,liquidated_co...",2025-12-29 18:00:00.000 UTC,all_null(quiet/gap)=173 structural_zero=0 active=195
7,T2,flashloan: zero-coupling (count<=>amount<=>uniques),flashloan_tx_count,hard,368,368,0,100.0000,,,all_null(quiet/gap)=0 structural_zero=0 active=368
8,T2,liquidated collateral >= debt covered (USD; bonus),debt_covered <= collateral (usd),soft,195,193,2,98.9744,,2025-11-30 12:00:00.000 UTC; 2025-12-29 18:00:00.000 UTC,
9,T2,avg_ltv <= avg_current_liquidation_threshold,avg_ltv <= avg_current_liquidation_threshold,hard,289,289,0,100.0000,,,


### Tier 3 — Unit & cross-asset consistency
USD⇔ETH paired nullity across all amount pairs (consolidated); cross-family implied ETH price
coherence (≤ 5 % spread, soft); `avg_*_base` magnitude stability — a **series-level** test
(median `log10` of the first vs second half of the series; flags only a *sustained* ≥ 1-order
shift, i.e. a real decimals / base-unit break), soft. The earlier per-bucket-jump version was
dropped — it fired on ordinary small-sample swings (~50 % false positives), not unit breaks.

In [22]:
display(adv.tf_tier3_consistency(df_final))

,tier,check,columns,severity,n_checked,n_pass,n_fail,pass_rate_pct,anomaly_columns,anomaly_buckets,detail
0,T3,USD<=>ETH paired nullity (0/empty together),8 value pairs,hard,2598,2598,0,100.0,,,
1,T3,cross-family implied ETH price coherent (<= 5% spread),usd/eth across families,soft,368,368,0,100.0,,,median_spread=0.0000 p95=0.0002 max=0.0009
2,T3,avg_*_base magnitude stable (no 10^x base-unit break),avg_total_collateral_base,soft,1,1,0,100.0,,,median log10 1st-half=6.68 2nd-half=6.55 (Δ=-0.13 dex); range=[0.7...
3,T3,avg_*_base magnitude stable (no 10^x base-unit break),avg_total_debt_base,soft,1,1,0,100.0,,,median log10 1st-half=6.37 2nd-half=6.47 (Δ=+0.10 dex); range=[-1....
4,T3,avg_*_base magnitude stable (no 10^x base-unit break),avg_available_borrows_base,soft,1,1,0,100.0,,,median log10 1st-half=6.11 2nd-half=5.74 (Δ=-0.38 dex); range=[-0....


### Tier 4 — Temporal integrity
`time_bucket` unique (no duplicate PK); lands on the 6h grid (00/06/12/18 UTC); **no gaps**
— full 6h coverage over the range, any missing buckets enumerated in `anomaly_buckets`
(distinguishing a gap from a quiet bucket); strictly increasing in row order.

In [23]:
display(adv.tf_tier4_temporal(df_final))

,tier,check,columns,severity,n_checked,n_pass,n_fail,pass_rate_pct,anomaly_columns,anomaly_buckets,detail
0,T4,time_bucket unique (no duplicate PK),time_bucket,hard,368,368,0,100.0,,,distinct=368 of 368 rows
1,T4,time_bucket lands on a 6h boundary (00/06/12/18 UTC),time_bucket,hard,368,368,0,100.0,,,
2,T4,no gaps: full 6h coverage over min..max range,time_bucket,hard,368,368,0,100.0,,,expected=368 present=368 missing=0 range=[2025-11-01 00:00 .. 2026...
3,T4,time_bucket strictly increasing in row order,time_bucket,hard,367,367,0,100.0,,,


### `DF_common_1` — reserve panel (applicable subset)
`data_val.md` targets the protocol-level frame; the asset-level reserve panel gets the
translatable checks: composite-PK `(time_bucket, asset)` uniqueness, time schema / 6h-grid,
rate non-negativity, `index ≥ 1.0`, and per-asset index monotonicity (interest only accrues up).

In [ ]:
display(adv.validate_reserve_panel(df_panel, save=False))

,tier,check,columns,severity,n_checked,n_pass,n_fail,pass_rate_pct,anomaly_columns,anomaly_buckets,detail
0,T0/T4,"composite PK (time_bucket, asset) unique","time_bucket,asset",hard,6164,6164,0,100.0,,,distinct keys=6164 of 6164 rows
1,T0,time_bucket is a tz-explicit UTC timestamp,time_bucket,hard,6164,6164,0,100.0,,,
2,T4,time_bucket lands on a 6h boundary,time_bucket,hard,6164,6164,0,100.0,,,
3,T1,rate >= 0 (RAY->decimal),last_borrow_rate,hard,5446,5446,0,100.0,,,observed_min=2.5344613483365408e-08
4,T1,rate >= 0 (RAY->decimal),liquidity_rate,hard,6164,6164,0,100.0,,,observed_min=0.0
5,T1,rate >= 0 (RAY->decimal),variable_borrow_rate,hard,6164,6164,0,100.0,,,observed_min=1.40928992225625e-08
6,T1,index >= 1.0 (RAY->decimal),liquidity_index,hard,6164,6164,0,100.0,,,observed_min=1.0
7,T2,index non-decreasing per asset over time,liquidity_index,hard,6128,6128,0,100.0,,,
8,T1,index >= 1.0 (RAY->decimal),variable_borrow_index,hard,6164,6164,0,100.0,,,observed_min=1.0000000638446662
9,T2,index non-decreasing per asset over time,variable_borrow_index,hard,6128,6128,0,100.0,,,


### Consolidated — save tidy results to `validation_results/`
Runs every tier and writes one tidy CSV per frame
(`DF_common_final__transform_validation.csv`, `DF_common_1__transform_validation.csv`).
The overview counts failing checks per **frame / tier / severity**; **soft** checks
(liquidation economics, solvency, implied-price spread, magnitude stability) are advisory.

In [ ]:
final_res = adv.validate_transformed_final(df_final)   # saves DF_common_final__transform_validation.csv
panel_res = adv.validate_reserve_panel(df_panel)       # saves DF_common_1__transform_validation.csv

# overview: failing checks per frame / tier / severity (soft checks are advisory)
both = pd.concat([final_res.assign(frame="DF_common_final"),
                  panel_res.assign(frame="DF_common_1")], ignore_index=True)
overview = (both.assign(failing=both["n_fail"] > 0)
            .groupby(["frame", "tier", "severity"], as_index=False)
            .agg(checks=("check", "size"), failing_checks=("failing", "sum")))
print(f"DF_common_final: {len(final_res)} checks  |  DF_common_1: {len(panel_res)} checks")
print("saved -> validation_results/DF_common_final__transform_validation.csv")
print("saved -> validation_results/DF_common_1__transform_validation.csv")
display(overview)

DF_common_final: 30 checks  |  DF_common_1: 10 checks
saved -> validation_results/DF_common_final__transform_validation.csv
saved -> validation_results/DF_common_1__transform_validation.csv


,frame,tier,severity,checks,failing_checks
0,DF_common_1,T0,hard,1,0
1,DF_common_1,T0/T4,hard,1,0
2,DF_common_1,T1,hard,5,0
3,DF_common_1,T2,hard,2,0
4,DF_common_1,T4,hard,1,0
5,DF_common_final,T0,hard,4,1
6,DF_common_final,T1,hard,4,1
7,DF_common_final,T2,hard,10,1
8,DF_common_final,T2,soft,3,2
9,DF_common_final,T3,hard,1,0
